# Lab 04 - Frame It Three Ways, Then Guard the Contract

**Week 3 · Prompt Engineering and Task-to-Prompt Mapping**
**Difficulty:** Intermediate · **Time:** ~150 minutes · **Runs fully offline (no network, no API key).**

You are helping Finance Ops at a fictional retailer, **Fernwood Hardware**, triage a week of
travel-expense messages and produce a manager digest. One vague stakeholder request:

> "Can you look at last week's travel expense emails, sort them, pull the key fields, and give me a quick summary for managers?"

That single sentence hides **three different tasks**. In this lab you will:

1. **Frame it three ways** - classification, extraction (schema-first), and summarization - each as a precise, delimited prompt with an explicit output contract.
2. **Guard the contract** - encode the extraction contract as a JSON Schema, validate model output, generate a repair instruction from the errors, and repair to a clean result.
3. **Report verified numbers** - build the manager digest from the clean data.

**Why offline.** Live model output is non-reproducible, so this lab replaces the network call with a
deterministic `simulate_model(...)` stub that returns realistic, contract-violating output. The engineering
you practice (framing, schema guardrails, validate-repair loops) is exactly what you would wrap around a
real endpoint. An appendix shows how to swap the stub for a live Anthropic or OpenAI call.

**How to work this notebook.** Cells with `# TODO` are yours to complete. Each is followed by a `check(...)`
cell. The notebook opens mostly **red** and is done when every check reads **PASS**. Hints are deliberately
sparse; if you get stuck, open `HINTS.md`.

## Part A - Corpus and Model Stub (provided)

Run the next two cells as-is. They define the policy, the six messages, the reference schema fields,
and the deterministic model stub. You do not edit these.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from importlib.metadata import version
print("jsonschema", version("jsonschema"))

POLICY = """1. Eligible categories: airfare, lodging, ground_transport, meals, other.
2. Receipts required for any single expense >= 25.00 USD.
3. Tips are allowed up to 20 percent and must be itemized.
4. For foreign currency, record original currency and USD converted total.
5. Missing receipt: mark status 'needs_proof' and request re-submission."""

SNIPPETS = {
 "E-101": "Took Uber from SFO to hotel, USD 42.85, tip 15%, receipt attached.",
 "E-102": "Hotel in Austin 3 nights, total $612.44. I think the receipt is somewhere.",
 "E-103": "Flight AA1421 SFO to AUS $389.60. Fare + taxes included. PDF receipt included.",
 "E-104": "Team dinner: $128.90 for 4 people at 'Oak and Rye'. Forgot to keep receipt.",
 "E-105": "Metro card in Austin about $25. Cash. No receipt, sorry.",
 "E-106": "Taxi from AUS to client site 38.00 USD, no tip. Photo of receipt included.",
}
VALID_CATEGORIES = ["airfare", "lodging", "ground_transport", "meals", "other"]
ALLOWED_KEYS = {"input_id","category","amount_usd","tip_percent","currency",
                "has_receipt","status","original_currency","notes"}
CATEGORY_ALIASES = {"hotel":"lodging","flight":"airfare","air":"airfare","cab":"ground_transport",
                    "taxi":"ground_transport","uber":"ground_transport","rideshare":"ground_transport",
                    "food":"meals","dinner":"meals"}
print("corpus loaded:", len(SNIPPETS), "messages")

In [ ]:
import copy

# Deterministic stand-in for "paste this prompt into your model and read back JSON".
# The first pass returns realistic, contract-violating output. A stronger (few-shot)
# prompt slips fewer flaws. This is the ONLY thing standing in for a live model.
_MODEL_TRUTH = [
 {"input_id":"E-101","category":"ground_transport","amount_usd":42.85,"tip_percent":15.0,
  "currency":"USD","has_receipt":True,"status":"ok","original_currency":None,
  "notes":"Uber to hotel; receipt attached; tip within policy."},
 {"input_id":"E-102","category":"lodging","amount_usd":612.44,"tip_percent":None,
  "currency":"USD","has_receipt":False,"status":"needs_proof","original_currency":None,
  "notes":"Hotel 3 nights; receipt uncertain; needs proof."},
 {"input_id":"E-103","category":"airfare","amount_usd":389.60,"tip_percent":None,
  "currency":"USD","has_receipt":True,"status":"ok","original_currency":None,
  "notes":"Flight; PDF receipt included."},
 {"input_id":"E-104","category":"meals","amount_usd":128.90,"tip_percent":None,
  "currency":"USD","has_receipt":False,"status":"needs_proof","original_currency":None,
  "notes":"Team dinner; no receipt kept; needs proof."},
 {"input_id":"E-105","category":"ground_transport","amount_usd":25.00,"tip_percent":None,
  "currency":"USD","has_receipt":False,"status":"needs_proof","original_currency":None,
  "notes":"Metro card; cash; no receipt; at 25 threshold."},
 {"input_id":"E-106","category":"ground_transport","amount_usd":38.00,"tip_percent":None,
  "currency":"USD","has_receipt":True,"status":"ok","original_currency":None,
  "notes":"Taxi to client site; photo receipt; no tip."},
]

def simulate_model(task, few_shot=False):
    """Deterministic offline model. task='extract' returns a list of records.
    Zero-shot output carries six realistic flaws; few_shot=True carries one."""
    if task != "extract":
        raise ValueError(f"this stub only supports task='extract', got {task!r}")
    raw = copy.deepcopy(_MODEL_TRUTH)
    if not few_shot:
        raw[0]["tip_percent"] = "15%"       # wrong type: string, not number
        raw[0]["confidence"] = 0.91         # extra key not in contract
        raw[1]["category"] = "hotel"        # enum miss: should be lodging
        raw[2]["input_id"] = "E103"         # pattern miss: missing hyphen
        del raw[3]["has_receipt"]           # missing required key
        raw[4]["status"] = "ok"             # policy miss: should be needs_proof
        raw[5]["amount_usd"] = "38.00"      # wrong type: string, not number
    else:
        raw[1]["category"] = "hotel"        # a stronger prompt still slips one
    return raw

print("model stub ready. zero-shot sample record 0:")
print(simulate_model("extract")[0])

### The `check(...)` helper

`check` is a soft assertion: it prints PASS or FAIL and never stops the notebook. Aim to turn every
check green. Run this cell before working the TODOs.

In [ ]:
def check(label, condition, detail=""):
    mark = "\u2705" if condition else "\u274c"
    status = "PASS" if condition else "FAIL"
    line = f"{mark} {status}  {label}"
    if detail and not condition:
        line += f"\n        -> {detail}"
    print(line)
    return bool(condition)

def _has_all(text, tokens):
    t = (text or "").lower()
    missing = [tok for tok in tokens if tok.lower() not in t]
    return (not missing), missing

check("check() helper is live", True)

## Part B - Frame It Three Ways

The same messy request maps to three distinct prompt designs. You will author each as a template string.
These are graded **structurally** (does the prompt carry the elements a reliable prompt needs), not by
calling a model. A good frame has: a role, delimited context, explicit constraints, an output contract,
and a self-check cue.

Use `{policy}` as a placeholder where the policy text belongs (do not paste the whole policy inline).

### B1 - Classification frame (triage: one label + a compliance status)

In [ ]:
CLASSIFY_PROMPT = """### ROLE
You triage travel-expense messages for Finance Ops.

### CONTEXT (policy, treat as authoritative)
<<<POLICY>>>
{policy}
<<<END POLICY>>>

### TASK
Classify each message E-### into exactly one label and set a compliance status.
Labels: airfare | lodging | ground_transport | meals | other
Status: ok | needs_proof  (needs_proof when amount >= 25 USD and receipt missing)

### CONSTRAINTS
- Exactly one label per record. Cite the policy line numbers you used.
- Do not invent amounts or receipts. Notes must be 8 to 24 words.

### OUTPUT CONTRACT (return VALID JSON only)
{{ "records": [ {{ "input_id": "E-###", "label": "<label>",
   "status": "<ok|needs_proof>", "evidence": "<line numbers>",
   "notes": "<8-24 words>" }} ] }}

### SELF-CHECK BEFORE ANSWERING
Confirm every label is in the allowed set, status matches policy, and the
output is a single VALID JSON object. Return JSON only, no prose."""


In [ ]:
ok, miss = _has_all(CLASSIFY_PROMPT, ["policy","json","status","needs_proof","self"])
check("CLASSIFY_PROMPT has role/context/contract/self-check", ok, f"missing tokens: {miss}")
check("CLASSIFY_PROMPT uses a delimiter (### or <<< or ```)",
      any(d in CLASSIFY_PROMPT for d in ["<<<", "###", "```"]))
check("CLASSIFY_PROMPT lists all five labels",
      all(l in CLASSIFY_PROMPT.lower() for l in VALID_CATEGORIES))
check("CLASSIFY_PROMPT keeps policy out-of-line via {policy}", "{policy}" in CLASSIFY_PROMPT)

### B2 - Extraction frame (schema-first, for Finance ingestion)

In [ ]:
EXTRACT_PROMPT = """### ROLE
You extract structured expense records for Finance ingestion.

### CONTEXT (policy)
<<<POLICY>>>
{policy}
<<<END POLICY>>>

### TASK
For each message E-###, extract fields per the schema. Output a JSON array.

### SCHEMA (fields, types, required)
input_id (string, required) | category (enum, required) |
amount_usd (number, required) | tip_percent (number or null) |
currency (string or null) | has_receipt (boolean, required) |
status (enum ok|needs_proof, required) | original_currency (string or null) |
notes (string, <= 160 chars)

### RULES
- Do not invent values. Numbers are numeric (no currency symbols or percent signs).
- has_receipt is true only if a receipt/photo/PDF is stated.
- Return a VALID JSON array only, no prose, no extra keys.

### SELF-VERIFY
Before answering, confirm every required key is present, types match, and
input_id matches E-###. Emit JSON array only."""


In [ ]:
ok, miss = _has_all(EXTRACT_PROMPT, ["schema","json","invent","required","self"])
check("EXTRACT_PROMPT has schema/contract/no-invention/self-verify", ok, f"missing tokens: {miss}")
check("EXTRACT_PROMPT uses a delimiter",
      any(d in EXTRACT_PROMPT for d in ["<<<", "###", "```"]))
check("EXTRACT_PROMPT keeps policy out-of-line via {policy}", "{policy}" in EXTRACT_PROMPT)

### B3 - Summarization frame (manager digest)

In [ ]:
SUMMARIZE_PROMPT = """### ROLE
You write a manager-facing weekly expense digest for Finance managers.

### SCOPE
Summarize messages E-101..E-106: total spend, category split, and items
needing proof. Neutral tone. No PII, no blaming language.

### STYLE / LENGTH
At most 6 bullets, at most 14 words each.

### OUTPUT CONTRACT (Markdown)
### Weekly Travel Expense Digest
- Total spend (USD): <number>
- Category split: <comma-separated>
- Items needing proof: <E-ids or 'none'>
- Policy risks: <up to 2 bullets>
- Quick actions (max 2): <bulleted>

### SELF-CHECK
Confirm the total equals the sum of amount_usd and that flagged items match
the needs_proof set. Return the digest only."""


In [ ]:
ok, miss = _has_all(SUMMARIZE_PROMPT, ["manager","total spend","proof","bullet"])
check("SUMMARIZE_PROMPT targets managers with a digest contract", ok, f"missing tokens: {miss}")
check("SUMMARIZE_PROMPT constrains length (mentions 6 and 14)",
      ("6" in SUMMARIZE_PROMPT and "14" in SUMMARIZE_PROMPT))

## Part C - Guard the Contract (schema, validate, repair)

Prompts alone do not guarantee well-formed output; a capable model still returns the occasional bad record.
So we enforce the contract in code. You will build the schema, a validator, a repair-prompt generator, and a
deterministic repair step, then run the validate-repair-revalidate loop against the stub's flawed output.

### C1 - JSON Schema (Draft 2020-12)

Encode the extraction contract as a Draft 2020-12 schema for an **array of objects**.

In [ ]:
EXPENSE_SCHEMA = {
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "array",
  "items": {
    "type": "object",
    "required": ["input_id", "category", "amount_usd", "has_receipt", "status"],
    "properties": {
      # "input_id": {"type": "string", "pattern": "^E-\\d{3}$"},
      "input_id": {"type": "string", "pattern": "^E-[0-9]{3}$"},
      "category": {"type": "string", "enum": VALID_CATEGORIES},
      "amount_usd": {"type": "number", "minimum": 0},
      "tip_percent": {"type": ["number", "null"], "minimum": 0, "maximum": 20},
      "currency": {"type": ["string", "null"]},
      "has_receipt": {"type": "boolean"},
      "status": {"type": "string", "enum": ["ok", "needs_proof"]},
      "original_currency": {"type": ["string", "null"]},
      "notes": {"type": ["string", "null"], "maxLength": 160}
    },
    "additionalProperties": False
  }
}


In [ ]:
from jsonschema import Draft202012Validator
try:
    Draft202012Validator.check_schema(EXPENSE_SCHEMA)
    check("EXPENSE_SCHEMA is itself a valid Draft 2020-12 schema", True)
    good = [{"input_id":"E-999","category":"meals","amount_usd":10.0,
             "has_receipt":True,"status":"ok"}]
    bad_extra = [dict(good[0], nope=1)]
    v = Draft202012Validator(EXPENSE_SCHEMA)
    check("schema accepts a well-formed record", v.is_valid(good))
    check("schema rejects an unexpected extra key", not v.is_valid(bad_extra))
    check("schema rejects a bad input_id pattern",
          not v.is_valid([dict(good[0], input_id="E999")]))
    check("schema rejects an out-of-range tip_percent",
          not v.is_valid([dict(good[0], tip_percent=150)]))
except Exception as e:
    check("EXPENSE_SCHEMA usable", False, repr(e))

### C2 - Validator

Wrap the schema in a function that returns a **sorted list of readable error strings** (empty when valid).

In [ ]:
def validate(data, schema=None):
    schema = schema if schema is not None else EXPENSE_SCHEMA
    v = Draft202012Validator(schema)
    errors = sorted(v.iter_errors(data), key=lambda e: e.json_path)
    return [f"{e.json_path}: {e.message}" for e in errors]


In [ ]:
try:
    raw = simulate_model("extract")
    errs = validate(raw)
    check("validate() returns a list", isinstance(errs, list))
    check("validate() flags the six seeded structural errors on raw output",
          len(errs) == 6, f"got {len(errs)}: {errs}")
    check("validate() returns [] for a clean record",
          validate([{"input_id":"E-100","category":"other","amount_usd":1.0,
                     "has_receipt":True,"status":"ok"}]) == [])
    print("\nseeded errors:")
    for e in errs: print("  ", e)
except NotImplementedError as e:
    check("validate() implemented", False, str(e))
except Exception as e:
    check("validate() runs without crashing", False, repr(e))

> **Notice what is *not* in that list.** The stub set `E-105` to `status: "ok"`, which is a legal enum
> value, so the schema passes it. Schema validity is not the same as policy correctness. The repair step
> below re-derives status from policy and catches it.

### C3 - Repair-prompt generator

If you were sending output back to a real model to fix, you would hand it the errors plus the contract rules.
Build that instruction text from a list of errors.

In [ ]:
def make_repair_prompt(errors, schema_name="schemas/expense.schema.json"):
    joined = "\n".join(errors) if errors else "(none)"
    return (
        "You are a strict JSON repair assistant.\n"
        f"CONTRACT: output must conform to {schema_name} (Draft 2020-12).\n"
        "Rules:\n"
        "- Do not delete valid records; fix fields or add missing required keys.\n"
        "- Enforce enums and types; coerce tip_percent to a number in 0..20 or null.\n"
        "- input_id must match ^E-###; category must be in the allowed enum.\n"
        "- Recompute status from policy: needs_proof if amount_usd >= 25 and no receipt, else ok.\n"
        "- Keep only contract keys; no additional properties.\n"
        "- Return a VALID JSON array ONLY, no prose.\n\n"
        "Validator errors to fix:\n---\n"
        f"{joined}\n---\n"
        "Emit ONLY the corrected JSON array."
    )


In [ ]:
try:
    p = make_repair_prompt(validate(simulate_model("extract")))
    check("repair prompt demands a JSON array only", "VALID JSON array ONLY" in p.upper() or "JSON ARRAY ONLY" in p.upper())
    check("repair prompt states the policy status rule", "needs_proof" in p)
    check("repair prompt embeds the validator errors", "additional properties" in p.lower())
    check("empty errors embed '(none)'", "(none)" in make_repair_prompt([]))
except NotImplementedError as e:
    check("make_repair_prompt() implemented", False, str(e))
except Exception as e:
    check("make_repair_prompt() runs", False, repr(e))

### C4 - Deterministic repair

In this offline lab, apply the repair in code (the same transforms the repair prompt asks the model to make).
This also teaches that many guardrails need no second model call at all.

In [ ]:
import re
def _to_number(val):
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return float(val)
    s = re.sub(r"[^0-9.\-]", "", str(val))
    return float(s) if s not in ("", "-", ".") else None

def repair_records(raw):
    fixed = []
    for rec in raw:
        r = {k: v for k, v in rec.items() if k in ALLOWED_KEYS}
        digits = re.sub(r"\D", "", str(r.get("input_id", "")))
        r["input_id"] = f"E-{digits[-3:].zfill(3)}" if digits else r.get("input_id")
        r["amount_usd"] = _to_number(r.get("amount_usd"))
        r["tip_percent"] = _to_number(r.get("tip_percent"))
        cat = str(r.get("category", "")).strip().lower()
        r["category"] = CATEGORY_ALIASES.get(cat, cat)
        r["has_receipt"] = bool(r.get("has_receipt", False))
        amt = r["amount_usd"] or 0.0
        r["status"] = "needs_proof" if (amt >= 25.0 and not r["has_receipt"]) else "ok"
        fixed.append(r)
    return fixed


In [ ]:
try:
    raw = simulate_model("extract")
    clean = repair_records(raw)
    check("repair_records() does not mutate its input", "confidence" in raw[0])
    check("repaired output passes the schema", validate(clean) == [],
          f"still invalid: {validate(clean)}")
    check("E-102 category alias hotel -> lodging",
          next(r for r in clean if r["input_id"]=="E-102")["category"] == "lodging")
    check("E-103 input_id normalized to E-103",
          any(r["input_id"]=="E-103" for r in clean))
    check("E-105 status corrected to needs_proof by policy",
          next(r for r in clean if r["input_id"]=="E-105")["status"] == "needs_proof")
    check("E-106 amount coerced to numeric 38.0",
          next(r for r in clean if r["input_id"]=="E-106")["amount_usd"] == 38.0)
except NotImplementedError as e:
    check("repair_records() implemented", False, str(e))
except Exception as e:
    check("repair_records() runs", False, repr(e))

## Part D - Manager Digest and the Full Loop

Now consume the clean records to produce the manager numbers, then run the whole pipeline end to end.

In [ ]:
def build_manager_summary(clean):
    total = round(sum(r["amount_usd"] for r in clean), 2)
    seen, split = set(), []
    for r in clean:
        if r["category"] not in seen:
            seen.add(r["category"]); split.append(r["category"])
    needs = [r["input_id"] for r in clean if r["status"] == "needs_proof"]
    return {"total_spend_usd": total, "category_split": split, "needs_proof": needs}


In [ ]:
try:
    summary = build_manager_summary(repair_records(simulate_model("extract")))
    check("total spend is the verified 1236.79", summary["total_spend_usd"] == 1236.79,
          f"got {summary['total_spend_usd']}")
    check("category split is correct and de-duplicated",
          summary["category_split"] == ["ground_transport","lodging","airfare","meals"],
          f"got {summary['category_split']}")
    check("needs_proof set is E-102, E-104, E-105",
          summary["needs_proof"] == ["E-102","E-104","E-105"], f"got {summary['needs_proof']}")
    print("\nsummary:", summary)
except NotImplementedError as e:
    check("build_manager_summary() implemented", False, str(e))
except Exception as e:
    check("build_manager_summary() runs", False, repr(e))

### The full validate-repair-revalidate loop

This provided cell ties everything together the way a production guardrail would.

In [ ]:
def run_pipeline(few_shot=False, verbose=True):
    raw = simulate_model("extract", few_shot=few_shot)
    before = validate(raw)
    repair_prompt = make_repair_prompt(before)          # what we'd send a live model
    clean = repair_records(raw)                          # what we do offline
    after = validate(clean)
    summary = build_manager_summary(clean)
    if verbose:
        print(f"errors before repair: {len(before)}")
        print(f"errors after repair:  {len(after)}")
        print("digest:", summary)
    return {"before": before, "after": after, "clean": clean, "summary": summary,
            "repair_prompt": repair_prompt}

def triage_view(clean):
    return [{"input_id":r["input_id"],"label":r["category"],"status":r["status"]} for r in clean]

try:
    result = run_pipeline()
    check("pipeline ends with zero validator errors", result["after"] == [])
    check("pipeline reduced the error count", len(result["before"]) > len(result["after"]))
    print("\ntriage view:")
    for row in triage_view(result["clean"]): print("  ", row)
except Exception as e:
    check("run_pipeline() runs end to end", False, repr(e))

## Stretch Goals (optional)

Both have graded checks. Solutions ship in the instructor solution notebook.

### Stretch 1 - Does a stronger prompt need fewer repairs?

The stub accepts `few_shot=True` to emulate a better-engineered prompt. Quantify the difference: count how
many validator errors each variant produces before repair, and confirm both still repair to the same clean
digest.

In [ ]:
def compare_prompt_variants():
    z = validate(simulate_model("extract", few_shot=False))
    f = validate(simulate_model("extract", few_shot=True))
    # sanity: both repair to the same verified digest
    for fs in (False, True):
        s = build_manager_summary(repair_records(simulate_model("extract", few_shot=fs)))
        assert s["total_spend_usd"] == 1236.79
    return {"zero_shot_errors": len(z), "few_shot_errors": len(f)}


In [ ]:
try:
    cmp = compare_prompt_variants()
    check("zero-shot produces more pre-repair errors than few-shot",
          cmp["zero_shot_errors"] > cmp["few_shot_errors"], f"got {cmp}")
    print("comparison:", cmp)
except NotImplementedError as e:
    check("compare_prompt_variants() implemented", False, str(e))
except Exception as e:
    check("compare_prompt_variants() runs", False, repr(e))

### Stretch 2 - A second guardrail layer: policy checks the schema cannot express

A schema cannot say "status must match the policy computed from amount and receipt." Build that second
layer. It should return violations for schema-valid-but-policy-wrong records and pass clean data.

In [ ]:
def policy_violations(records):
    out = []
    for r in records:
        amt = r.get("amount_usd") or 0.0
        expected = "needs_proof" if (amt >= 25.0 and not r.get("has_receipt")) else "ok"
        if r.get("status") != expected:
            out.append(f"{r.get('input_id')}: status '{r.get('status')}' but policy says '{expected}'")
        tp = r.get("tip_percent")
        if tp is not None and not (0 <= tp <= 20):
            out.append(f"{r.get('input_id')}: tip_percent {tp} outside 0..20")
    return out


In [ ]:
try:
    schema_ok_policy_wrong = [{"input_id":"E-201","category":"meals","amount_usd":50.0,
        "has_receipt":False,"status":"ok"}]
    check("that record passes the schema", validate(schema_ok_policy_wrong) == [])
    pv = policy_violations(schema_ok_policy_wrong)
    check("policy layer flags the schema-valid-but-policy-wrong record", len(pv) == 1, f"got {pv}")
    check("clean pipeline data has zero policy violations",
          policy_violations(repair_records(simulate_model("extract"))) == [])
    print("policy violations on the crafted record:", pv)
except NotImplementedError as e:
    check("policy_violations() implemented", False, str(e))
except Exception as e:
    check("policy_violations() runs", False, repr(e))

## Part F - Calling a real model with Ollama (`gemma4`): direct vs. system

Everything above used a deterministic stub. Now we call a real local model, `gemma4`, through Ollama, and
contrast the two ways engineers reach for an LLM:

- **Direct call** - a user hands the model one freeform message and reads back whatever prose comes out. Fast
  to write, impossible to trust downstream. There is no contract, so there is nothing to validate.
- **System around the LLM** - the model is one component inside a pipeline. A `system` prompt carries the
  role, the policy, and the output contract; the user turn carries only the data; generation is constrained
  to a JSON Schema; and the output flows into the same validate, repair, and policy layers you built in Parts
  C and D. The model becomes a drop-in replacement for `simulate_model`, and the guardrail does not change.

This section runs live against `gemma4` when an Ollama server is reachable. When it is not (for example in a
grading sandbox or a machine without the model pulled), it falls back to the deterministic stub so the
notebook still executes end to end and every check stays green. The point being taught is the framework, not
the specific tokens the model returns.

> Verified at build time against `ollama.com/library/gemma4` and `docs.ollama.com`: `gemma4` is a current
> Ollama tag (`gemma4:latest` is the e4b build, 9.6GB, 128K context, with native `system` role support), and
> Ollama structured outputs are requested by passing a JSON Schema to `format=` on `ollama.chat(...)`, with
> the response text in `response.message.content`.

In [ ]:
import os, json, urllib.request

# Keep the model id in a config variable; never hard-code an unverified id. Confirm the tag is
# pulled on your box (`ollama pull gemma4`) before class.
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "gemma4")
OLLAMA_HOST  = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

def ollama_available(host=OLLAMA_HOST, timeout=1.5):
    """True only if the ollama client imports AND a server answers. Any failure returns False,
    which routes the notebook onto the deterministic fallback."""
    try:
        with urllib.request.urlopen(host, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

OLLAMA_LIVE = ollama_available()
print("ollama reachable:", OLLAMA_LIVE, "| model:", OLLAMA_MODEL)
print("live path calls gemma4" if OLLAMA_LIVE else "offline: using deterministic cached fallback")

In [ ]:
# Cached DIRECT reply for offline runs: realistic, chatty, and deliberately NOT machine-parseable,
# which is exactly the problem with the direct pattern.
_CACHED_DIRECT_REPLY = (
    "Sure! Here is a quick rundown of those expenses:\n"
    "- A couple of rides, a hotel stay, a flight, a team dinner, and a metro card.\n"
    "- A few are missing receipts, so those likely need follow up.\n"
    "- The rough total lands somewhere around twelve hundred dollars.\n"
    "Want this as a table or split out by person?"
)

def call_llm_direct(user_text):
    """DIRECT path: one freeform user turn. No system role, no schema, no validation.
    Returns whatever text the model produced. This is 'a user calling the model'."""
    if OLLAMA_LIVE:
        from ollama import chat
        resp = chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": user_text}])
        return resp.message.content
    return _CACHED_DIRECT_REPLY

def call_llm_system(messages, schema, options=None):
    """SYSTEM path: system + user turns, schema-constrained generation, deterministic options.
    Returns a parsed Python object (list of records). Falls back to the lab's deterministic
    stub when offline, so the guardrail below runs identically in any environment."""
    options = options or {"temperature": 0}   # lower temp for schema adherence; see Part E flag
    if OLLAMA_LIVE:
        from ollama import chat
        resp = chat(model=OLLAMA_MODEL, messages=messages,
                    format=schema, options=options)
        return json.loads(resp.message.content)
    return simulate_model("extract")   # cached, deterministic, carries the same six flaws

print("call_llm_direct and call_llm_system are defined")


In [ ]:
# print("EXPENSE_SCHEMA:\n", json.dumps(EXPENSE_SCHEMA, indent=2))
# print("EXTRACT_PROMPT:\n", EXTRACT_PROMPT)
# print("\nPOLICY:\n", POLICY)
print("\nCOMBINATION:\n", EXTRACT_PROMPT.format(policy=POLICY))

In [ ]:
# Build both request shapes from artifacts you already wrote.
corpus_text = "\n".join(f"{k}: {v}" for k, v in SNIPPETS.items())

# DIRECT: the naive ask. Everything in one user string, no contract.
# direct_user = (
#     "Here are some travel expense notes. Sort them, pull the key fields, "
#     "and give me a manager summary.\n\n" + corpus_text
# )
# direct_reply = call_llm_direct(direct_user)
# print("=== DIRECT path output (freeform, not pipeline-safe) ===")
# print(direct_reply)

# SYSTEM: the engineered ask. The system turn carries role + policy + contract (reuse EXTRACT_PROMPT);
# the user turn carries only the data. Generation is constrained to EXPENSE_SCHEMA.
system_messages = [
    {"role": "system", "content": EXTRACT_PROMPT.format(policy=POLICY)},
    {"role": "user",   "content": corpus_text},
]
print("Corpus Text:", corpus_text)
# print("EXTRACT_PROMPT:", EXTRACT_PROMPT)
# print("POLICY:", POLICY)
# print("COMBINATION:", EXTRACT_PROMPT.format(policy=POLICY))
print("System Messages:", system_messages)
llm_records = call_llm_system(system_messages, EXPENSE_SCHEMA, options={"temperature": 0})

# The punchline: feed model output into the SAME guardrail, unchanged.
errs_before = validate(llm_records)
clean_llm   = repair_records(llm_records)
errs_after  = validate(clean_llm)
policy_llm  = policy_violations(clean_llm)
summary_llm = build_manager_summary(clean_llm)

print("\n=== SYSTEM path through the Part C-D guardrail ===")
print("errors before repair:", len(errs_before))
print("errors after repair: ", len(errs_after))
print("policy violations remaining:", policy_llm)
print("digest:", summary_llm)

In [ ]:
print(llm_records)

In [ ]:
print("--- Part F checks ---")
check("SYSTEM path: guardrail clears every schema error",
      len(errs_after) == 0, f"{len(errs_after)} remain")
check("SYSTEM path: digest total is the verified 1236.79",
      summary_llm["total_spend_usd"] == 1236.79, str(summary_llm["total_spend_usd"]))
check("SYSTEM path: needs_proof flags E-104, E-105",
      summary_llm["needs_proof"] == ["E-104", "E-105"], str(summary_llm["needs_proof"]))
check("SYSTEM path: no residual policy violations",
      policy_llm == [], str(policy_llm))
# check("DIRECT path returns prose, not a parseable record list",
#       not _is_record_list(direct_reply))

## Part F (variant) - The same guardrail, LM Studio instead of Ollama

Only the **transport** changes. LM Studio exposes an **OpenAI-compatible** server, so instead of
`ollama.chat(..., format=schema)` you call `client.chat.completions.create(..., response_format=json_schema)`
with the `openai` client pointed at LM Studio's local endpoint. The system/user turns, the contract, and the
Part C-D validate → repair → policy loop are identical; the model is still a drop-in for `simulate_model`.

Two LM Studio specifics to note:
- Its structured-output `response_format` wants an **object at the schema root**, so we wrap the array
  contract as `{"records": [...]}` for generation and unwrap the result.
- The `api_key` is required by the client but **ignored** by the local server (use any placeholder).

If LM Studio is not running, these cells fall back to the deterministic stub so the notebook still executes.


In [ ]:
# LM Studio ships an OpenAI-compatible server (default http://localhost:1234/v1). Steps:
#   1. In LM Studio, download a chat model that supports structured output (e.g. gpt-oss / Qwen / Gemma).
#   2. Open the "Developer" / "Local Server" tab and Start Server (load the model there).
#   3. pip install openai  (the client speaks LM Studio's OpenAI-compatible API).
# We auto-detect the loaded chat model from /v1/models so this runs against whatever you have loaded;
# override with the LMSTUDIO_MODEL env var if you want a specific one.
import os, json, urllib.request

LMSTUDIO_BASE = os.environ.get("LMSTUDIO_BASE", "http://localhost:1234/v1")

def lmstudio_models(base=LMSTUDIO_BASE, timeout=1.5):
    """Return the model ids the LM Studio server currently exposes ([] if unreachable).
    An empty list routes this section onto the deterministic fallback."""
    try:
        import openai  # noqa: F401
        with urllib.request.urlopen(base.rstrip("/") + "/models", timeout=timeout) as r:
            data = json.loads(r.read())
        return [m["id"] for m in data.get("data", [])]
    except Exception:
        return []

_MODELS = lmstudio_models()
_chat_models = [m for m in _MODELS if "embed" not in m.lower()]   # skip embedding models
LMSTUDIO_MODEL = os.environ.get("LMSTUDIO_MODEL") or (_chat_models[0] if _chat_models else "your-model-id")
LMSTUDIO_LIVE = bool(_chat_models)

print("LM Studio reachable:", LMSTUDIO_LIVE, "| model:", LMSTUDIO_MODEL)
print("available models:", _MODELS)
print("live path calls LM Studio" if LMSTUDIO_LIVE else "offline: using deterministic cached fallback")


In [ ]:
# LM Studio's structured output uses the OpenAI response_format=json_schema shape, which requires
# an OBJECT at the schema root. Our contract is an array, so we wrap it as {"records": [...]} for
# GENERATION and unwrap on the way out. Validation below still uses the ORIGINAL array EXPENSE_SCHEMA,
# so the same guardrail runs unchanged.
#
# LM Studio's engine is llama.cpp/GBNF (same family as Ollama), so it also cannot compile the
# `pattern` regex; we reuse _ollama_safe_schema (defined earlier) to strip grammar-only-incompatible
# keywords for GENERATION, and re-check them downstream with the full schema.
def _as_object_schema(array_schema):
    """Wrap an array-of-objects JSON Schema into an object schema with one 'records' array."""
    return {
        "type": "object",
        "properties": {"records": {"type": "array", "items": array_schema.get("items", {})}},
        "required": ["records"],
        "additionalProperties": False,
    }

def call_llm_direct_lmstudio(user_text):
    """DIRECT path against LM Studio: one freeform user turn, no schema, no validation.
    Returns whatever prose the model produced (or the cached reply when offline)."""
    if LMSTUDIO_LIVE:
        from openai import OpenAI
        client = OpenAI(base_url=LMSTUDIO_BASE, api_key="lm-studio")  # key is ignored locally
        resp = client.chat.completions.create(
            model=LMSTUDIO_MODEL,
            messages=[{"role": "user", "content": user_text}],
        )
        return resp.choices[0].message.content
    return _CACHED_DIRECT_REPLY

def call_llm_system_lmstudio(messages, schema, options=None):
    """SYSTEM path against LM Studio's OpenAI-compatible endpoint. Returns a list of records.
    Falls back to the deterministic stub when LM Studio is unreachable, so the guardrail below
    runs identically in any environment.

    response_format=json_schema is LM Studio's structured-output constraint; it shares the SAME
    contract you validate with (wrapped as an object root, pattern stripped for the grammar). A
    constrained model can still omit a required field, so downstream validation is not optional."""
    options = options or {"temperature": 0}
    if LMSTUDIO_LIVE:
        from openai import OpenAI
        client = OpenAI(base_url=LMSTUDIO_BASE, api_key="lm-studio")
        resp = client.chat.completions.create(
            model=LMSTUDIO_MODEL,
            messages=messages,
            temperature=options.get("temperature", 0),
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "expense_records",
                    "strict": True,
                    "schema": _as_object_schema(schema),
                },
            },
        )
        payload = json.loads(resp.choices[0].message.content)
        return payload["records"] if isinstance(payload, dict) and "records" in payload else payload
    return simulate_model("extract")   # cached, deterministic, carries the same six flaws

print("call_llm_direct_lmstudio and call_llm_system_lmstudio are defined")


In [ ]:
# Same system_messages you built for the Ollama path; only the transport changes.
lms_records = call_llm_system_lmstudio(system_messages, EXPENSE_SCHEMA, options={"temperature": 0})

# The punchline is unchanged: feed model output into the SAME Part C-D guardrail.
errs_before_lms = validate(lms_records)
clean_lms       = repair_records(lms_records)
errs_after_lms  = validate(clean_lms)
policy_lms      = policy_violations(clean_lms)
summary_lms     = build_manager_summary(clean_lms)

print("=== LM Studio SYSTEM path through the Part C-D guardrail ===")
print("errors before repair:", len(errs_before_lms))
print("errors after repair: ", len(errs_after_lms))
print("policy violations remaining:", policy_lms)
print("digest:", summary_lms)

print("lms_records:", lms_records)

In [ ]:
print("--- Part F (Variant) checks ---")
check("SYSTEM path: guardrail clears every schema error",
      len(errs_after_lms) == 0, f"{len(errs_after_lms)} remain")
check("SYSTEM path: digest total is the verified 1236.79",
      summary_lms["total_spend_usd"] == 1236.79, str(summary_lms["total_spend_usd"]))
check("SYSTEM path: needs_proof flags E-104, E-105",
      summary_lms["needs_proof"] == ["E-104", "E-105"], str(summary_lms["needs_proof"]))
check("SYSTEM path: no residual policy violations",
      policy_lms == [], str(policy_lms))

## Part E - Wrap-Up

### Knowledge checks (discuss)
1. Why split this one request into classification, extraction, and summarization rather than asking for everything at once?
2. What did the JSON Schema catch that a hand-written `if` ladder would likely miss, and what did it *not* catch?
3. When would you re-prompt a model to repair versus fix the data in code? What are the trade-offs?

### Self-assessment rubric (0-2 each, target >= 10/12)
- Three delimited frames, each with a role, constraints, and an output contract.
- Schema encodes types, enums, pattern, and rejects extra keys.
- Validator returns readable, sorted errors and is empty on clean data.
- Repair prompt fully restates the contract and embeds the errors.
- Deterministic repair yields schema-clean, policy-correct records.
- Manager digest reports the verified total and the correct flags.

### CURRENCY FLAG - determinism controls
> `temperature=0` and `top_p=1` **reduce** variation but do **not** guarantee identical output on hosted
> endpoints; a `seed` is best-effort and provider-specific (OpenAI exposes one; the Anthropic Messages API
> does not), and reasoning models lock sampling parameters. Treat schema validation plus a repair loop as
> your real determinism guarantee, not the sampling knobs. Verify the current parameter surface for your
> provider at build time.
>
> **Model-specific note (`gemma4`):** its model card recommends `temperature=1.0, top_p=0.95, top_k=64` for general use. For schema-constrained extraction you will usually lower temperature (Ollama's structured-output guidance suggests `0`) to improve adherence, which deliberately departs from the general-use recommendation. Constrained generation can still omit a required field, so the validate-repair loop stays in place regardless.

### Live model: see Part F
Part F replaces the old reference-only appendix. It calls `gemma4` through Ollama when a server is
reachable and falls back to the deterministic stub otherwise, so the notebook runs anywhere. The
model output flows into the exact same `validate`, `repair_records`, `policy_violations`, and
`build_manager_summary` loop you built in Parts C and D.

In [ ]:
print("Lab complete when every check above reads PASS.")